# 🛡️ Comment Toxicity Detection - Deep Learning
**Model:** Bidirectional LSTM | **Output:** 6 Toxicity Classes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, LSTM, Bidirectional, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
print(f"TensorFlow: {tf.__version__}")

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('train.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Label Distribution
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
df[labels].sum().plot(kind='bar', color='steelblue', title='Label Distribution')
plt.savefig('label_distribution.png')
plt.show()

## 2. Preprocessing

In [ ]:
X = df['comment_text'].values
y = df[labels].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

In [ ]:
# Text Vectorization
MAX_TOKENS = 200000
SEQ_LEN = 1800

vectorizer = TextVectorization(max_tokens=MAX_TOKENS, output_sequence_length=SEQ_LEN)
vectorizer.adapt(X_train)
print(f"Vocab Size: {len(vectorizer.get_vocabulary())}")

## 3. Build & Train Model

In [ ]:
model = Sequential([
    vectorizer,
    Embedding(MAX_TOKENS + 1, 128),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(6, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Train
history = model.fit(X_train, y_train, epochs=1, batch_size=32, validation_data=(X_val, y_val))

## 4. Evaluate & Save

In [ ]:
# Evaluate
loss, acc = model.evaluate(X_val, y_val)
print(f"\nVal Loss: {loss:.4f}, Val Accuracy: {acc:.4f}")

In [ ]:
# Save Model
model.save('toxicity_model_end_to_end.h5')

# Save History
import json
with open('training_history.json', 'w') as f:
    json.dump(history.history, f)

print("✅ Model Saved!")

## 5. Test Prediction

In [ ]:
# Test
test_texts = ["You are amazing!", "I hate you, go die!"]
preds = model.predict(test_texts)

for text, pred in zip(test_texts, preds):
    print(f"\n'{text}'")
    for label, score in zip(labels, pred):
        print(f"  {label}: {score:.1%}")